### Retiro Fugas

In [1]:
port = '1433'

user_zeus = "Zeus"
pwd_zeus = "target12345"
server_zeus = "192.168.2.12"
db_zeus = "THOTH"

user_sa = "sa"
pwd_sa = "target2023$"
server_sa = "192.168.2.50"
db_sa = "CRONOX"

user_kishin = "kishin"
pwd_kishin = "Leto0891.1"
server_kishin = "192.168.2.15"
db_kishin = "DANTALION"

In [22]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-06-01'

filename='efectiva.xlsx'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)

df = df.rename(columns={
    f'{name_dni}': 'NUMERO_DOCUMENTO'
})
df=df[["NUMERO_DOCUMENTO"]]
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

query = f"""
    SELECT DISTINCT DNI as NUMERO_DOCUMENTO
    FROM (
        select DNI
        from DANTALION.dbo.Base_Maestra_Efectiva_Vigente
        WHERE RETIRO IS NULL  OR RETIRO =''
        UNION
        select NUMDOCUMENTO
        from DANTALION.dbo.Base_Maestra_Efectiva_Negocios_Vigente
        WHERE RETIRO IS NULL  OR RETIRO =''
    ) t
    """
df_efe = pd.read_sql(query, engine_kishin)

df = df.merge(df_efe, on='NUMERO_DOCUMENTO', how='inner')
list_dni = (
    df['NUMERO_DOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)
# in_clause = ",".join(f"'{x}'" for x in list_dni)
valores = ",\n".join(f"('{dni}')" for dni in list_dni)


In [28]:
if not list_dni:
    print('terminar')

terminar


In [ ]:
from sqlalchemy import text

try:
    with engine_kishin.begin() as conn:

        query_tmp = f"""
        CREATE TABLE #tmp_dni (
            DNI VARCHAR(20)
        );

        INSERT INTO #tmp_dni (DNI)
        VALUES
        {valores};

        CREATE INDEX IX_tmp_dni ON #tmp_dni(DNI);
        """

        conn.execute(text(query_tmp))

        query_1 = f"""
        DECLARE @fecha_inicio DATE = TRY_CONVERT(DATE, '{fecha_mes_base}');
        DECLARE @fecha_fin DATE = DATEADD(MONTH, 1, @fecha_inicio);

        UPDATE A
        SET a.RETIRO = 'RETIRO'
        FROM DANTALION.dbo.Base_Maestra_Efectiva_Negocios A
        INNER JOIN #tmp_dni B
            ON A.NUMDOCUMENTO = B.DNI COLLATE Modern_Spanish_CI_AS
        WHERE TRY_CONVERT(DATE, A.fecha_envio) >= @fecha_inicio
          AND TRY_CONVERT(DATE, A.fecha_envio) < @fecha_fin;
        """

        result_1 = conn.execute(text(query_1))
        print("Negocios:", result_1.rowcount)

        query_2 = f"""
        DECLARE @fecha_inicio DATE = TRY_CONVERT(DATE, '{fecha_mes_base}');
        DECLARE @fecha_fin DATE = DATEADD(MONTH, 1, @fecha_inicio);

        UPDATE A
        SET a.RETIRO = 'RETIRO'
        FROM DANTALION.dbo.Base_Maestra_Efectiva A
        INNER JOIN #tmp_dni B
            ON A.DNI = B.DNI COLLATE Modern_Spanish_CI_AS
        WHERE TRY_CONVERT(DATE, A.fecha_envio) >= @fecha_inicio
          AND TRY_CONVERT(DATE, A.fecha_envio) < @fecha_fin;
        """

        result_2 = conn.execute(text(query_2))
        print("Efectiva:", result_2.rowcount)

except Exception as e:
    print("Error:", e)

Error: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Sintaxis incorrecta cerca de ';'. (102) (SQLExecDirectW)")
[SQL: 
        CREATE TABLE #tmp_dni (
            DNI VARCHAR(20)
        );

        INSERT INTO #tmp_dni (DNI)
        VALUES
        ;

        CREATE INDEX IX_tmp_dni ON #tmp_dni(DNI);
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [19]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva_Negocios", "SP tNumeros Efectiva_Negocios")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar Efectiva_Negocios Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar Efectiva_Negocios SA")

SP tNumeros Efectiva_Negocios | realizado | duración: 236.24 seg
SP actualizar Efectiva_Negocios Zeus | realizado | duración: 32.63 seg
SP actualizar Efectiva_Negocios SA | realizado | duración: 16.14 seg


In [20]:

exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva", "SP tNumeros Efectiva")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva", "SP actualizar Efectiva Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva", "SP actualizar Efectiva SA")

SP tNumeros Efectiva | realizado | duración: 56.67 seg
SP actualizar Efectiva Zeus | realizado | duración: 15.15 seg
SP actualizar Efectiva SA | realizado | duración: 37.57 seg


In [8]:
df.count()

NUMERO_DOCUMENTO    18
dtype: int64

In [9]:
try:
    with engine_kishin.begin() as conn:

        query_1 = f"""
        DECLARE @fecha_inicio DATE = TRY_CONVERT(DATE, '{fecha_mes_base}');
        DECLARE @fecha_fin DATE = DATEADD(MONTH, 1, @fecha_inicio);

        UPDATE DANTALION.dbo.Base_Maestra_Efectiva_Negocios
        SET RETIRO = 'RETIRO'
        WHERE NUMDOCUMENTO IN ({in_clause})
          AND TRY_CONVERT(DATE, fecha_envio) >= @fecha_inicio
          AND TRY_CONVERT(DATE, fecha_envio) < @fecha_fin;
        """

        result_1 = conn.execute(text(query_1))
        print("Filas actualizadas negocios:", result_1.rowcount)

        query_2 = f"""
        DECLARE @fecha_inicio DATE = TRY_CONVERT(DATE, '{fecha_mes_base}');
        DECLARE @fecha_fin DATE = DATEADD(MONTH, 1, @fecha_inicio);

        UPDATE DANTALION.dbo.Base_Maestra_Efectiva
        SET RETIRO = 'RETIRO'
        WHERE DNI IN ({in_clause})
          AND TRY_CONVERT(DATE, fecha_envio) >= @fecha_inicio
          AND TRY_CONVERT(DATE, fecha_envio) < @fecha_fin;
        """

        result_2 = conn.execute(text(query_2))
        print("Filas actualizadas efectiva:", result_2.rowcount)

except Exception as e:
    print("Error:", e)

Filas actualizadas negocios: 7
Filas actualizadas efectiva: 14


In [10]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners_tc", "SP tNumeros diners TC")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners_tc", "SP actualizar diners TC SA")

SP tNumeros diners TC | realizado | duración: 21.6 seg
SP actualizar diners TC Zeus | realizado | duración: 13.41 seg
SP actualizar diners TC SA | realizado | duración: 2.42 seg


In [ ]:


server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
fecha_mes_base='2026-06-01'
campana=fecha_a_nombre('2026-05-01')
query = f"""
select dni as NumDoc, 1 as venta_target from SAMANTHA.dbo.Ventas_Target
where CAMPANA='Diners'
and CONVERT(DATE, FECHA) >= CONVERT(DATE, '{fecha_mes_base}')
AND CONVERT(DATE, FECHA) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
   
"""
df_venta = pd.read_sql(query, engine_zeus)

df_target = df_tc.merge(
    df_venta,
    on='NumDoc',
    how='left'
)
df_final = df_target.merge(
    df,
    on='NumDoc',
    how='inner'
)
df_final = (
    df_final[df_final['venta_target'].isnull()]
    .drop(columns=['venta_target'])
)
df_final.rename(
    columns={
        'Importe Solicitado': 'Monto'
    },
    inplace=True
)
print(df_final.columns.tolist())

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


['NumDoc', 'TIPO_PRODUCTO', 'RETIRO', 'Canal', 'Autor', 'Subcanal', 'Motivo', 'Monto', 'fecha', 'hora']


In [72]:
df_final=df_final[df_final['Monto'].notnull()]

In [78]:
df_final[['TIPO_PRODUCTO','Canal', 'Monto', 'fecha']].head()


,TIPO_PRODUCTO,Canal,Monto,fecha
0,PPD,CANALES DIGITALES,41100.0,2026-06-04
1,PPD,CANALES DIGITALES,13800.0,2026-06-04
2,PPD,CONTACT CENTER,6000.0,2026-06-04


In [ ]:

total_monto_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].sum()

total_ope_ppd = df_final.loc[
    df_final['TIPO_PRODUCTO'] == 'PPD',
    'Monto'
].count()

print(f'PPD | Monto total: {int(total_monto_ppd)} |  Total operaciones {int(total_ope_ppd)} ')

PPD | Monto total: 60900 |  Total operaciones 3 


In [ ]:
# ruta_archivo = os.path.join(ruta_csv, 'PPD_plus.xlsx')
# df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
list_dni = (
    df_final.loc[df_final['RETIRO'].isna(), 'NumDoc']
    .dropna()
    .drop_duplicates()
    .tolist()
)

in_clause = ",".join(f"'{x}'" for x in list_dni)

try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE DANTALION.dbo.Base_Maestra_Diners
            SET RETIRO = 'RETIRO'
            WHERE NumDoc IN ({in_clause})
                and CONVERT(DATE, fecha_envio) >= CONVERT(DATE, '{fecha_mes_base}')
                AND CONVERT(DATE, fecha_envio) < DATEADD(MONTH, 1, CONVERT(DATE, '{fecha_mes_base}'))
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)


In [77]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.48 seg
SP actualizar diners Zeus | realizado | duración: 5.54 seg
SP actualizar diners SA | realizado | duración: 1.5 seg
